<a href="https://colab.research.google.com/github/ambika-1513/Computer-vision-learning/blob/main/midas_video_depth_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
torch.cuda.is_available()

True

In [3]:
torch.cuda.device_count()

1

In [4]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [5]:
!pip install timm -q

In [15]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import timm

In [8]:
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.to("cuda")
midas.eval()

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None
The repository rwightman_gen-efficientnet-pytorch does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:00<00:00, 304MB/s]


MidasNet_small(
  (pretrained): Module(
    (layer1): Sequential(
      (0): Conv2dSameExport(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
      (3): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (act1): ReLU6(inplace=True)
          (se): Identity()
          (conv_pw): Conv2d(32, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn2): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (act2): Identity()
        )
      )
      (4): Sequential(
        (0): InvertedResidual(
          (conv_pw): Conv2d(24, 144, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(144,

In [9]:
midas_transforms = torch.hub.load("intel-isl/MiDaS","transforms")
transform = midas_transforms.small_transform

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [10]:
cap = cv2.VideoCapture("traffic.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))



In [11]:
out = cv2.VideoWriter("output_depth.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps , (width*2, height))

In [12]:
frame_num =0

In [16]:
while True:
  ret, frame = cap.read()
  if not ret:
    break
  frame_num +=1
  img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  img_batch = transform(img_rgb).to("cuda")
  with torch.no_grad():
    prediction = midas(img_batch)
    prediction = torch.nn.functional.interpolate(prediction.unsqueeze(1),
                                                 size = img_rgb.shape[:2],
                                                 mode = "bicubic",
                                                 align_corners =False).squeeze()
  depth_map = prediction.cpu().numpy()
  depth_norm = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
  depth_colored = cv2.applyColorMap(depth_norm, cv2.COLORMAP_INFERNO)
  combined = np.hstack((frame, depth_colored))
  out.write(combined)
  if frame_num % 30 == 0:
    print(f"Frame {frame_num} processed")
cap.release()
out.release()

Frame 30 processed
Frame 60 processed
Frame 90 processed
Frame 120 processed
Frame 150 processed
Frame 180 processed
Frame 210 processed
Frame 240 processed
Frame 270 processed
Frame 300 processed
Frame 330 processed
Frame 360 processed
Frame 390 processed
Frame 420 processed
Frame 450 processed
Frame 480 processed
Frame 510 processed
Frame 540 processed
Frame 570 processed
Frame 600 processed
Frame 630 processed
Frame 660 processed
Frame 690 processed
Frame 720 processed
Frame 750 processed
Frame 780 processed
Frame 810 processed
Frame 840 processed
Frame 870 processed
Frame 900 processed
Frame 930 processed
Frame 960 processed
Frame 990 processed
Frame 1020 processed
Frame 1050 processed
Frame 1080 processed
Frame 1110 processed
Frame 1140 processed
Frame 1170 processed
Frame 1200 processed
Frame 1230 processed
Frame 1260 processed
Frame 1290 processed
Frame 1320 processed
Frame 1350 processed
Frame 1380 processed
Frame 1410 processed
Frame 1440 processed
Frame 1470 processed
Frame 1